# BananaQ CNN — corrected notebook
Run cells in order in Google Colab with the dataset on Drive. For local Jupyter, install TensorFlow, NumPy, Pillow, matplotlib and scikit-learn, then set BASE_DIR below.

The dataset must contain train/, val/ and test/ with black_sigatoka/, cordana/, healthy/ and panama/ in each split. Existing extracted datasets are reused. Training, evaluation and export use the same best validation-loss checkpoint. Model inputs are RGB float32, shape (1, 224, 224, 3), scaled to [0, 1]; normalize exactly once in Android.

Old outputs have been cleared. This notebook still requires a training run; no accuracy is implied by these fixes.

In [ ]:
try:
    from google.colab import drive
except ImportError:
    print("Local Jupyter: configure BASE_DIR in the next cell.")
else:
    drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os
import json

BASE_DIR = Path('/content/drive/MyDrive') if Path('/content/drive/MyDrive').is_dir() else Path.cwd()
# For local use, edit BASE_DIR to the folder containing BananaQDataset.zip.
zip_path = BASE_DIR / 'BananaQDataset.zip'
extract_path = BASE_DIR / 'BananaQDataset'
output_dir = BASE_DIR / 'BananaQ'
output_dir.mkdir(parents=True, exist_ok=True)
best_model_path = str(output_dir / 'best_bananaq_model.keras')
tflite_path = str(output_dir / 'bananaq_model.tflite')
CLASS_NAMES = ['black_sigatoka', 'cordana', 'healthy', 'panama']
IMAGE_SIZE = (224, 224)
SEED = 42
print('Base directory:', BASE_DIR)

In [ ]:
import zipfile

def find_dataset_root(folder):
    candidates = [folder, folder / 'BananaQDataset']
    return next((p for p in candidates if all((p / s).is_dir() for s in ('train', 'val', 'test'))), None)

if find_dataset_root(extract_path) is None:
    if not zip_path.is_file():
        raise FileNotFoundError(f'Place BananaQDataset.zip in {BASE_DIR}, or extract its train/val/test folders into {extract_path}.')
    with zipfile.ZipFile(zip_path) as archive:
        destination = extract_path.resolve()
        for member in archive.infolist():
            if not (destination / member.filename).resolve().is_relative_to(destination):
                raise ValueError(f'Unsafe archive path: {member.filename}')
        archive.extractall(extract_path)
    print('Dataset extracted.')
else:
    print('Using existing extracted dataset.')

In [ ]:
dataset_path = find_dataset_root(extract_path)
if dataset_path is None:
    raise FileNotFoundError(f'Expected train/, val/ and test/ under {extract_path} or its BananaQDataset subfolder.')
for split in ('train', 'val', 'test'):
    actual = {p.name for p in (dataset_path / split).iterdir() if p.is_dir() and not p.name.startswith('.')}
    if actual != set(CLASS_NAMES):
        raise ValueError(f'{split}: expected class folders {CLASS_NAMES}, found {sorted(actual)}')
print('Dataset:', dataset_path)

In [ ]:
train_path = dataset_path / 'train'
print('Classes:', CLASS_NAMES)

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models

print("TensorFlow version:", tf.__version__)

tf.keras.utils.set_random_seed(SEED)

In [ ]:
train_dir = str(dataset_path / 'train')
val_dir = str(dataset_path / 'val')
test_dir = str(dataset_path / 'test')

train_datagen = ImageDataGenerator(
    rescale=1./255,

    rotation_range=30,

    width_shift_range=0.15,
    height_shift_range=0.15,

    shear_range=0.15,

    zoom_range=0.25,

    horizontal_flip=True,

    brightness_range=[0.7, 1.3]
)

val_datagen = ImageDataGenerator(
    rescale=1./255
)


train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    classes=CLASS_NAMES,
    interpolation='nearest',
    seed=SEED
)


val_data = val_datagen.flow_from_directory(
    val_dir,
    shuffle=False,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    classes=CLASS_NAMES,
    interpolation='nearest',
    seed=SEED
)

print(train_data.class_indices)

if train_data.samples == 0 or val_data.samples == 0:
    raise ValueError('Training and validation splits must contain supported images.')
for iterator in (train_data, val_data):
    if set(iterator.classes.tolist()) != set(range(len(CLASS_NAMES))):
        raise ValueError('Each class must contain at least one supported image in every split.')
labels = [name.replace('_', ' ').title() for name in CLASS_NAMES]
(output_dir / 'labels.txt').write_text('\n'.join(CLASS_NAMES) + '\n', encoding='utf-8')

In [ ]:
model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),

    layers.Conv2D(
        32,
        kernel_size=(3, 3),
        activation="relu"
    ),
    layers.MaxPooling2D(pool_size=(2, 2)),

    layers.Conv2D(
        64,
        kernel_size=(3, 3),
        activation="relu"
    ),
    layers.MaxPooling2D(pool_size=(2, 2)),

    layers.Conv2D(
        128,
        kernel_size=(3, 3),
        activation="relu"
    ),
    layers.MaxPooling2D(pool_size=(2, 2)),

    layers.Conv2D(
        256,
        kernel_size=(3, 3),
        activation="relu"
    ),
    layers.MaxPooling2D(pool_size=(2, 2)),

    layers.GlobalAveragePooling2D(),

    layers.Dense(
        128,
        activation="relu"
    ),

    layers.Dropout(0.4),

    layers.Dense(
        len(CLASS_NAMES),
        activation="softmax"
    )
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.callbacks import ReduceLROnPlateau

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True
    ),

    ModelCheckpoint(
        best_model_path,
        monitor="val_loss",
        save_best_only=True,
        mode="min"
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=0.00001
    )
]

print("Callbacks are ready.")

In [ ]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=20,
    callbacks=callbacks
)

In [ ]:
# Load the checkpoint selected during this training run; do not overwrite it.
if not Path(best_model_path).is_file():
    raise FileNotFoundError('Run the training cell to create the best checkpoint first.')
model = tf.keras.models.load_model(best_model_path)
print('Loaded best BananaQ checkpoint:', best_model_path)

In [ ]:
# Export the same checkpoint used for evaluation.
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
Path(tflite_path).write_bytes(tflite_model)
print('BananaQ TFLite model created:', tflite_path)

In [ ]:
print('Input shape:', model.input_shape)
print('Output shape:', model.output_shape)
model.summary()

In [ ]:
# Prefer LiteRT when installed; retain compatibility with TensorFlow runtimes.
try:
    from ai_edge_litert.interpreter import Interpreter
except ImportError:
    Interpreter = tf.lite.Interpreter

interpreter = Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
import numpy as np
assert tuple(input_details[0]['shape']) == (1, *IMAGE_SIZE, 3)
assert input_details[0]['dtype'] == np.float32
assert tuple(output_details[0]['shape']) == (1, len(CLASS_NAMES))
assert output_details[0]['dtype'] == np.float32
assert model.output_shape[-1] == len(CLASS_NAMES)
print('Verified TFLite input/output shapes and float32 types.')
print(dict(enumerate(CLASS_NAMES)))

In [ ]:
model = tf.keras.models.load_model(best_model_path)
test_datagen = ImageDataGenerator(rescale=1./255)
test_data = test_datagen.flow_from_directory(
    test_dir, target_size=IMAGE_SIZE, batch_size=32,
    class_mode='categorical', classes=CLASS_NAMES,
    interpolation='nearest', shuffle=False
)
if test_data.samples == 0 or set(test_data.classes.tolist()) != set(range(len(CLASS_NAMES))):
    raise ValueError('Each test class must contain supported images.')
test_loss, test_accuracy = model.evaluate(test_data)
print('Test loss:', test_loss)
print('Test accuracy:', test_accuracy)

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

test_data.reset()

predictions = model.predict(test_data)

predicted_classes = np.argmax(predictions, axis=1)
true_classes = test_data.classes

class_names = list(test_data.class_indices.keys())

print("Class mapping:")
print(test_data.class_indices)

print("\n===== CLASSIFICATION REPORT =====")

print(
    classification_report(
        true_classes,
        predicted_classes,
        target_names=CLASS_NAMES,
        labels=list(range(len(CLASS_NAMES))),
        zero_division=0
    )
)

print("\n===== CONFUSION MATRIX =====")

cm = confusion_matrix(
    true_classes,
    predicted_classes,
    labels=list(range(len(CLASS_NAMES)))
)

print(cm)

In [ ]:
from PIL import Image

def preprocess_image(image_path):
    # Match flow_from_directory's explicit nearest-neighbor resizing.
    image = tf.keras.utils.load_img(str(image_path), color_mode='rgb', target_size=IMAGE_SIZE, interpolation='nearest')
    array = tf.keras.utils.img_to_array(image).astype(np.float32) / 255.0
    return np.expand_dims(array, axis=0)

# Use an actual test file instead of assuming a particular filename exists.
image_path = Path(test_data.filepaths[0])
image_array = preprocess_image(image_path)
keras_output = model.predict(image_array, verbose=0)[0]
print('Image:', image_path)
for label, probability in zip(labels, keras_output):
    print(f'{label}: {probability * 100:.4f}%')

In [ ]:
interpreter.set_tensor(input_details[0]['index'], image_array)
interpreter.invoke()
tflite_output = interpreter.get_tensor(output_details[0]['index'])[0]
for label, probability in zip(labels, tflite_output):
    print(f'{label}: {probability * 100:.4f}%')
print('Maximum Keras/TFLite probability difference:', float(np.max(np.abs(keras_output - tflite_output))))
np.testing.assert_allclose(keras_output, tflite_output, rtol=1e-3, atol=1e-4)

In [ ]:
import random
rng = random.Random(SEED)

def predict_image(image_path):
    prediction = model.predict(preprocess_image(image_path), verbose=0)[0]
    index = int(np.argmax(prediction))
    print(f'{image_path}: {labels[index]} ({prediction[index] * 100:.2f}%)')
    return prediction

for class_name in CLASS_NAMES:
    images = [Path(p) for p in test_data.filepaths if Path(p).parent.name == class_name]
    if not images:
        raise ValueError(f'No supported test images for {class_name}')
    predict_image(rng.choice(images))

In [ ]:
import matplotlib.pyplot as plt

predicted_index = int(np.argmax(tflite_output))
predicted_class = labels[predicted_index]
confidence = float(tflite_output[predicted_index]) * 100
with Image.open(image_path) as img:
    plt.figure(figsize=(6, 6))
    plt.imshow(img.convert('RGB'))
    plt.axis('off')
    plt.title(f'{predicted_class} ({confidence:.2f}%)')
    plt.show()